# XGBoost (`booster`): proceso de construcción

Cuaderno de proceso del modelo de ensamble de árboles. Las celdas de código son lecturas
ligeras (no entrenan nada) y avisan si falta algún archivo. Modelos hermanos:
[`02_tcn.ipynb`](02_tcn.ipynb) y [`03_cnn_lstm.ipynb`](03_cnn_lstm.ipynb).

In [ ]:
import os
import warnings
from datetime import date
from pathlib import Path

import polars as pl
import yaml

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Raiz del repositorio:", ROOT)


def leer_csv(ruta):
    try:
        return pl.read_csv(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}. Ver la seccion 'Como reproducir'.")
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")


def leer_yaml(ruta):
    with open(ruta, encoding="utf-8") as archivo:
        return yaml.safe_load(archivo)

## 1. Objetivo y criterio de selección

Pronosticar el nivel diario del Orinoco en **Ciudad Bolívar** y **Palúa** a 1 a 14 días,
reportando en particular los **días 7 y 14** (instrucción del profesor). Este cuaderno cuenta
cómo se construyó el modelo **XGBoost (booster)**: qué se decidió, por qué y qué se descartó.

**Criterio de selección** (fijado antes de ver resultados):

1. `rel_mae < 1` en **cada** estación, donde `rel_mae` = MAE del modelo / MAE de la persistencia
   (Naive: repetir el último nivel observado) en los mismos cortes.
2. Entre los que cumplen, el menor MAE medio entre las dos estaciones.

**Por qué `rel_mae` y no MASE.** Resuelve el problema de comparar contra una referencia justa.
Por dentro, divide dos errores calculados sobre las mismas ventanas. Se descartó el MASE con
escala del historial de entrenamiento: esa escala puede dar valores mayores que 1 a la propia
persistencia si el período evaluado fue más difícil que el de entrenamiento, y entonces el
umbral "menor que 1" deja de significar algo. `rel_mae` sería la decisión equivocada si la
persistencia fuera casi perfecta (MAE cercano a cero), porque el cociente se vuelve inestable.

## 2. Datos

**Fuente:** `data/raw/DATOS RIO ORINOCO.xlsx`, hoja `Entrenar` (niveles diarios de Ayacucho,
Caicara, Ciudad Bolívar y Palúa), usando **solo 2009-01-01 a 2024-12-31** (período válido
según el profesor): 5 844 días por estación. Los archivos de `data/raw/` no se modifican.

**Qué ve el modelo.** El frame que reciben todos los runners es el parquet de
`TimeSeriesDataset` (`data/interim/orinoco_dataset.parquet`) limpiado con `clean_history`
([`features/dataset.py`](../src/orinoco_forecast/features/dataset.py)). Trae las **cuatro
estaciones como series** (`unique_id`), el nivel `y`, lluvia de NASA POWER (`rain_*`), ONI y
columnas de rezagos ya calculadas. XGBoost vuelve a rezagar todas las covariables históricas (14, 28, 56 y 90 días) y usa todas las columnas (`drop_covariates: []`).

### Limpieza (`clean_history`) y por qué

| Paso | Qué resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| Recortar a 2009-2024 | Datos fuera del período no son válidos | Filtra `ds` con `model_frame.start/end` de `configs/data.yaml` | Usar la serie desde 1974: más historia, pero el profesor la declaró no válida | Si se confirma que el período anterior sí es válido |
| Anular rachas planas (`suspect_flat`, 10 días o más iguales) | Un sensor detenido repite el mismo valor y enseñaría persistencia falsa | `label_quality` etiqueta y `clean_series` las pone en nulo | Dejarlas: contamina el aprendizaje | Si una racha plana fuera real (río en estiaje extremo) |
| Interpolar huecos de hasta 31 días | Los modelos no admiten nulos | Interpolación lineal por fecha (`max_gap_days: 31`); marca `imputed = True` | Rellenar con la media o con el último valor: distorsiona la dinámica | Si el hueco cubre una crecida: la línea recta la borra |
| Descartar los primeros 60 días de 2009 | Las columnas de rezagos (hasta `oni_lag_60`) aún no existen | `drop_nulls()` tras limpiar; la serie efectiva empieza el 2009-03-02 | Rellenar rezagos con ceros: inventa datos | Nunca es un error mientras no se invente nada |

En 2009-2024: 3 valores `missing` en Palúa (29 de febrero de 2012, 2016 y 2020), rachas planas
de Ayacucho (2019-03-07 a 2019-03-18) y Ciudad Bolívar (2020-03-24 a 2020-04-11), y un
`suspect_jump` de Palúa (2021-10-29) que se etiqueta pero no se modifica: **34 filas
interpoladas**. La última interpolada (2020-04-11) es anterior a todos los cortes de
validación, así que la interpolación no usa datos posteriores a un corte.

In [ ]:
try:
    from orinoco_forecast.evaluation.execution import load_history

    historial = load_history()
    resumen = (
        historial.group_by("unique_id")
        .agg(
            pl.len().alias("dias"),
            pl.col("ds").min().alias("desde"),
            pl.col("ds").max().alias("hasta"),
            pl.col("imputed").sum().alias("dias_imputados"),
        )
        .sort("unique_id")
    )
    print(resumen)
    print("Columnas que reciben los runners:", len(historial.columns))
    print(historial.columns)
except FileNotFoundError:
    print(
        "Falta data/interim/orinoco_dataset.parquet. Generalo con:\n"
        "  uv run python scripts/run_model.py booster --write-dataset"
    )
except Exception as error:
    print("No se pudo cargar el historial:", error)

## 3. Partición de los datos

| Bloque | Qué es |
|---|---|
| Entrenamiento | Todo `ds <= cutoff`; **crece** con cada corte. Cada corte reajusta el modelo. |
| Validación | Los 14 días siguientes a cada corte: **24 cortes** de 2021-01-26 a 2022-12-17, uno cada 30 días. |
| Test final | `ds >= 2023-01-01`. **Nunca se usó** para decidir nada. |

**Qué problema resuelve.** Una serie de tiempo no se puede barajar ni repartir al azar: el
modelo no debe ver el futuro de lo que predice. **Cómo funciona:** `walk_forward_splits`
([`data/splits.py`](../src/orinoco_forecast/data/splits.py)) genera los cortes hacia atrás
desde `test_start - (horizonte + 1)` y falla si una ventana invade el test. **Alternativa
descartada:** un reparto fijo tipo 70/15/15, que mide un solo período y no dice si el modelo
es consistente; con 24 cortes se mide la consistencia. **Sería un error** si hubiera pocos
datos (aquí 24 ventanas de 14 días son una muestra razonable, no grande).

El test se evalúa **una sola vez**, con el modelo que elija el equipo. Si se mira antes para
decidir, la predicción deja de ser genuina.

In [ ]:
try:
    from orinoco_forecast.data.splits import walk_forward_splits

    cfg = leer_yaml("configs/backtest.yaml")
    test_start = date.fromisoformat(str(cfg["test_start"]))
    fechas = historial.filter(pl.col("unique_id") == "ciudad_bolivar").sort("ds")["ds"].to_list()
    cortes = walk_forward_splits(
        fechas,
        test_start=test_start,
        horizon=cfg["horizon"],
        step_size=cfg["step_size"],
        n_windows=cfg["n_windows"],
    )
    print(len(cortes), "cortes:", cortes[0].cutoff, "a", cortes[-1].cutoff)
    print("La ultima validacion termina el", cortes[-1].validation_end, "| el test empieza el", test_start)
    assert cortes[-1].validation_end < test_start
    solapadas = [b.validation_start <= a.validation_end for a, b in zip(cortes, cortes[1:])]
    print("Ventanas de validacion solapadas:", any(solapadas))
except NameError:
    print("Ejecuta antes la celda de datos (falta el historial).")
except Exception as error:
    print("No se pudo calcular la particion:", error)

## 4. Diseño del modelo

Código: [`models/trees/tree.py`](../src/orinoco_forecast/models/trees/tree.py)
(`BoosterRunner`) y configuración [`configs/models/tree.yaml`](../configs/models/tree.yaml).
**Gradient boosting** construye cientos de árboles pequeños en secuencia: cada árbol corrige
el error que dejaron los anteriores y la predicción es la suma ponderada por la tasa de
aprendizaje. XGBoost es una implementación eficiente con regularización.

| Decisión | Qué problema resuelve | Cómo funciona por dentro | Alternativa descartada y por qué | Cuándo sería un error |
|---|---|---|---|---|
| **Estrategia directa**: un modelo por día del horizonte (14 modelos) | Pronosticar 14 días sin acumular error | `mlforecast` con `max_horizon=14` entrena el modelo del paso `h` para predecir `y(t+h)` desde información hasta `t` | Estrategia recursiva (usar la predicción del día 1 como entrada del día 2): el error se acumula (Ben Taieb et al. 2012) | Si el horizonte fuera muy largo: 14 modelos por estación se vuelven caros |
| **Diferencias** (`target_differences: [1]`): predecir el cambio diario | Un árbol no puede salir del rango visto: en niveles, una crecida récord quedaría recortada al máximo histórico | `mlforecast` modela `y_t - y_(t-1)` y reconstruye el nivel sumando | Predecir el nivel directamente: más simple pero acotado al rango de entrenamiento | Si la serie no tuviera tendencia ni crecidas extremas |
| **Rezagos del objetivo** 1, 2, 3, 7, 14, 21, 28 y 56 días | Capturar inercia diaria (1-3), ciclo semanal (7) y el retardo de 2 a 8 semanas de la planicie de inundación (14-56) | Cada rezago es una columna con el nivel de hace `k` días | Una ventana larga cruda de 56 columnas: más columnas sin más información | Si el retardo físico del río fuera mayor que 56 días |
| **Medias móviles** de 7 y 30 días sobre el rezago 1 | Suavizar el ruido diario y ver la tendencia del último mes | `RollingMean` sobre la serie ya rezagada | No incluirlas: el árbol debería reconstruirlas con muchos cortes | Si la serie cambiara de régimen en menos de una semana |
| **Covariables rezagadas** `covariate_lags: [14, 28, 56, 90]` (mínimo >= horizonte) | **Evitar fuga de datos** | Ver la demostración de abajo | No rezagar: el modelo del día 14 usaría un valor que aún no existe en el corte | Nunca es un error mientras se mantenga el mínimo >= horizonte |
| **`drop_covariates: []`**: ve todas las columnas | Decidir qué entradas usa cada modelo con validación, no a ojo | Lista de prefijos de columna ocultos (vacía: ve niveles, lluvia y ONI) | Quitar lluvia y ONI: probado, da lo mismo (sección 6) | Si el número de columnas creciera tanto que el árbol sobreajustara |
| **Semilla fija** `random_state: 42` | Reproducibilidad | `XGBRegressor` usa esa semilla en el submuestreo | Varias semillas: más caro; los árboles con `subsample` alto varían poco | Si se quisiera medir la variabilidad del modelo |

### Por qué las covariables van rezagadas al menos 14 días (anti-fuga)

Hallazgo verificado en `mlforecast` 1.1.0: con la estrategia directa, las covariables
dinámicas se alinean con la **fecha objetivo** `t+h`, no con el corte. Si se pasara el nivel
de Ayacucho sin rezagar, el modelo del día 14 usaría el valor de Ayacucho 14 días *después*
del corte, que no se conoce. Por eso `prelag_covariates` desplaza cada covariable `k` días
antes de dársela a `mlforecast` y el runner **falla** si el mínimo es menor que el horizonte.

In [ ]:
try:
    import polars as pl
    from datetime import date, timedelta

    from orinoco_forecast.models.trees.tree import BoosterRunner, prelag_covariates

    dias = [date(2020, 1, 1) + timedelta(days=i) for i in range(8)]
    juguete = pl.DataFrame({
        "unique_id": ["a"] * 8,
        "ds": dias,
        "y": [float(i) for i in range(8)],
        "aguas_arriba": [10.0 + i for i in range(8)],
    })
    rezagada = prelag_covariates(juguete, ["aguas_arriba"], [3])
    print("Con rezago 3, la columna del mismo dia desaparece y las primeras filas se descartan:")
    print(rezagada)

    runner = BoosterRunner()
    runner.params = {**runner.params, "covariate_lags": [7]}
    try:
        runner._prepare(juguete, {"horizon": 14})
    except ValueError as error:
        print("\nRezago minimo 7 con horizonte 14 -> error esperado:\n", error)
except Exception as error:
    print("No se pudo ejecutar la demostracion:", error)

In [ ]:
try:
    tree = leer_yaml("configs/models/tree.yaml")
    print("Rezagos del objetivo:", tree["lags"])
    print("Medias moviles:", tree["rolling_mean_windows"], "| diferencias:", tree["target_differences"])
    print("Rezagos de covariables:", tree["covariate_lags"], "| drop_covariates:", tree["drop_covariates"])
    print("Hiperparametros de XGBoost:")
    for clave, valor in tree["xgb_params"].items():
        print(f"  {clave}: {valor}")
except Exception as error:
    print("No se pudo leer configs/models/tree.yaml:", error)

## 5. Ajuste de hiperparámetros con Optuna

| Decisión | Qué problema resuelve | Cómo funciona | Alternativa descartada | Cuándo sería un error |
|---|---|---|---|---|
| **TPE** (*Tree-structured Parzen Estimator*) | Buscar buenos hiperparámetros con pocas pruebas | Separa las pruebas en "buenas" y "malas" y propone configuraciones donde las buenas son más densas | Búsqueda en rejilla o aleatoria: gastan el presupuesto en zonas malas | Con presupuesto enorme da casi lo mismo |
| **20 pruebas completas por modelo** | Comparar modelos sin favorecer a ninguno | Mismo presupuesto para XGBoost, TCN y CNN-LSTM | Dar más intentos al modelo favorito: mediría el esfuerzo, no el modelo | Si un modelo tuviera un espacio mucho más grande |
| **Objetivo: `rel_mae` medio** de las dos estaciones | Optimizar lo mismo que el criterio de selección | MAE del modelo / MAE de Naive, promediado entre estaciones | Optimizar el MAE absoluto: premiaría a la estación con errores mayores | Si una estación importara mucho más que la otra |
| **12 ventanas de ajuste en 2020**, anteriores a la validación | Que los hiperparámetros no se elijan con los datos de validación | Cortes del 2020-05-04 al 2021-01-12, cada 23 días | Usar los 24 cortes de validación: dejaría de ser una medición independiente | Si la hidrología de 2020 no representara los otros años |
| **Cada 23 días y no 30** | Evitar cortes cerca de filas interpoladas | El hueco imputado de Ciudad Bolívar (2020-03-24 a 2020-04-11) impide cortes a menos de 14 días | Mantener 30 días: obligaría a ventanas con datos inventados | Con datos sin huecos |

Espacio buscado: `n_estimators` 300 a 1500, `learning_rate` 0,01 a 0,15 (logarítmico),
`max_depth` 3 a 9, `subsample` 0,6 a 1,0, `colsample_bytree` 0,5 a 1,0, `min_child_weight`
1 a 20, `reg_lambda` 0,1 a 20 y tres combinaciones de rezagos de covariables.
El código de Optuna está en el commit `b87533d` de `main` (PR #7); el reajuste con datos
2009-2024 usó esa versión más una opción `--tag`.

In [ ]:
pruebas = leer_csv("reports/tuning/xgboost_trials.csv")
if pruebas is not None:
    print(pruebas["state"].value_counts())
    completas = pruebas.filter(pl.col("state") == "COMPLETE").sort("value")
    columnas = [c for c in ["number", "value", "rel_mae_ciudad_bolivar", "rel_mae_palua", "duration_s"] if c in completas.columns]
    with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
        print(completas.select(columnas).head(5))
    mejor = completas.row(0, named=True)
    print("Mejor prueba:", mejor["number"], "| rel_mae medio", round(mejor["value"], 4))
    print("Peor completa:", round(completas["value"].max(), 4))

## 6. Validación en 24 cortes

Promedios de los 24 cortes (2021-01-26 a 2022-12-17), contra Naive, MAE en metros. Fuente:
`run_model.py booster --against naive --folds` (reproduce estos números exactamente).

| Estación | Horizonte | Naive | XGBoost MAE | `rel_mae` |
|---|---|---|---|---|
| Ciudad Bolívar | día 7 | 0,526 | 0,279 | 0,53 |
| Ciudad Bolívar | día 14 | 1,064 | 0,444 | 0,42 |
| Ciudad Bolívar | días 1 a 14 | 0,570 | 0,278 | 0,49 |
| Palúa | día 7 | 0,435 | 0,360 | 0,83 |
| Palúa | día 14 | 0,891 | 0,548 | 0,62 |
| Palúa | días 1 a 14 | 0,488 | 0,341 | 0,70 |

**Cumple el criterio** (`rel_mae < 1` en ambas estaciones y en todos los horizontes).

### Significancia estadística (test de signos de `--folds`)

El promedio puede ocultar que el modelo gane en pocos cortes. El test de signos cuenta en
cuántos de los 24 cortes el modelo supera a Naive:

| Estación y horizonte | `sign_test_p` | Lectura |
|---|---|---|
| Ciudad Bolívar, pasos 1 a 14 | 0,0015 | ventaja clara |
| Ciudad Bolívar, día 7 | 0,0066 | ventaja clara |
| Ciudad Bolívar, día 14 | 0,00004 | ventaja clara |
| Palúa, día 14 | 0,0066 | ventaja clara |
| **Palúa, día 7** | **0,84** (gana en 54 % de los cortes) | **sin evidencia de que supere a Naive** |
| Palúa, pasos 1 a 14 | 0,064 (gana en 71 %) | en el límite |

Que el `rel_mae` medio sea menor que 1 no basta para afirmar una ventaja sistemática en
Palúa a 7 días.

### ¿Aportan la lluvia y el ONI?

Para el booster, sobre 24 cortes (`rel_mae` de los días 1 a 14):

| Entrada | Ciudad Bolívar | Palúa |
|---|---|---|
| Todas las columnas (lo que se usa) | 0,487 | 0,698 |
| Solo niveles | 0,504 | 0,684 |

La diferencia es de uno o dos centésimos y cambia de signo según la estación: los árboles
ignoran bien las columnas que no sirven. Se conserva `drop_covariates: []`. (Con el TCN la
conclusión es distinta: ver `02_tcn.ipynb`.)

In [ ]:
def comparacion(runner):
    ruta = ROOT / f"data/predictions/{runner}/backtest/naive/comparison.parquet"
    try:
        tabla = pl.read_parquet(ruta)
    except FileNotFoundError:
        print(f"No existe {ruta}.\nGeneralo con: uv run python scripts/run_model.py {runner} --against naive --folds")
        return None
    except Exception as error:
        print(f"No se pudo leer {ruta}: {error}")
        return None
    try:
        columnas = ["unique_id", "model", "horizon", "mean_model", "mean_baseline",
                    "mean_improvement_pct", "win_rate", "sign_test_p"]
        return (
            tabla.filter((pl.col("metric") == "mae") & pl.col("horizon").cast(pl.String).is_in(["7", "14", "all"]))
            .select(columnas)
            .sort("unique_id", "model", "horizon")
        )
    except Exception as error:
        print("El archivo existe pero no tiene el formato esperado:", error)
        return tabla


resultado = comparacion("booster")
if resultado is not None:
    with pl.Config(tbl_cols=-1, tbl_width_chars=200, float_precision=4):
        print(resultado)

## 7. Limitaciones y siguientes pasos

- **Reajuste por corte vs Optuna.** `--folds` reentrena en cada corte; Optuna eligió los
  hiperparámetros con un ajuste único y 12 ventanas de 2020. Los parámetros siguen siendo
  válidos, pero la medición no es idéntica.
- **Sin Fourier.** El parquet no trae `sin_*`/`cos_*` (`calendar_terms: 0`), así que el modelo
  no ve el ciclo anual de forma explícita. No se midió cuánto cambiaría.
- **Entrenan con las cuatro estaciones.** `execute` pasa las cuatro series y filtra la
  predicción a las dos objetivo. No se probó entrenar solo con las dos objetivo.
- **NSE negativo.** Cada ventana dura 14 días y su varianza interna es pequeña, así que el
  NSE castiga cualquier error (también a Naive). Se interpreta `rel_mae`.
- **Una validación de 24 cortes es una muestra moderada.** Los resultados se leen junto con el
  test de signos, no solo con el promedio.
- **Test final sin tocar.** Estos resultados son de validación y pueden ser algo optimistas.
- **Sin datos de la represa de Guri** (regula el Caroní, cerca de Palúa).
- **Importancia de variables sin recalcular** para la configuración actual: la versión anterior
  (datos 1974+) estaba dominada por medias móviles y por niveles de Caicara y Ayacucho rezagados.
- **Una prueba abierta** en el CSV de Optuna (estado `RUNNING`): no cuenta entre las 20 completas.

**Siguientes pasos:** probar Fourier, entrenar solo con las dos estaciones objetivo, y decidir con
el equipo el protocolo de la evaluación final (una sola ejecución con todos los modelos).

## 8. Cómo reproducir

```bash
uv sync
# Genera el parquet (una vez)
uv run python scripts/run_model.py booster --write-dataset
# Un corte contra Naive
uv run python scripts/run_model.py booster --against naive
# 24 cortes (salidas en data/predictions/booster/backtest/naive/)
uv run python scripts/run_model.py booster --against naive --folds
```

Para usar GPU en Colab: [`colab_folds.ipynb`](colab_folds.ipynb) (cambia la variable
`MODEL`). Los resultados en GPU pueden variar un poco respecto a CPU porque el runner no activa
las opciones de determinismo de cuDNN.